In [1]:
import io
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

In [71]:
s = '''
Пользователь	Интерстеллар	Звездный войны	Дюна	Побег из Шоушенка	Зеленая миля	1+1
Алиса	3	5		1	1	
Боб	4	5	5	2	1	2
Чарли	3	5	4	2		3
Мэллори	5	5	4	2		3
Чак	1	1		5	5	
Дейв	3	3	2	3	5	5
Ева	1	1	2	5	4	4
Пэгги				5	5	5
'''

df = pd.read_csv(io.StringIO(s.strip()), sep='\t', index_col='Пользователь')
display(df)
df_filled = df.fillna(0)
user_similarity_df = pd.DataFrame(
     cosine_similarity(df_filled), index=df.index, columns=df.index
)
display(user_similarity_df)
display(user_similarity_df.loc['Алиса'].sort_values(ascending=False))

def get_recs_coll_filtering(user, df):
    similarity_matrix = pd.DataFrame(
     cosine_similarity(df_filled), index=df.index, columns=df.index
    )
    # Оценки целевого пользователя
    user_ratings = df.loc[user]
    
    # Фильмы, которые пользователь еще НЕ оценивал
    unrated_movies = user_ratings[user_ratings.isna()].index
    
    recommendations = {}
    
    for movie in unrated_movies:
        # Оценки других пользователей для этого фильма
        movie_ratings = df[movie]
        
        # Пользователи, которые оценили этот фильм
        rated_users = movie_ratings.dropna().index
        
        if len(rated_users) > 0:
            # Веса (похожесть) только тех пользователей, кто оценил фильм
            weights = similarity_matrix.loc[user, rated_users]
            
            # Оценки этих пользователей
            scores = movie_ratings.loc[rated_users]
            
            # Прогнозируем оценку как взвешенное среднее
            predicted_score = np.dot(scores, weights) / weights.sum()
            recommendations[movie] = predicted_score
            
    # Сортируем фильмы по прогнозируемой оценке от лучшего к худшему
    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs

def get_rec_result(name):
    print(f'Рекомендации коллаборативной фильтрации для {name}')
    recs = get_recs_coll_filtering(name, df)
    for movie, score in recs:
        print(f"\t{movie}: прогнозируемая оценка = {score:.2f}")
    print('\n')

get_rec_result('Алиса')
get_rec_result('Чак')

,Интерстеллар,Звездный войны,Дюна,Побег из Шоушенка,Зеленая миля,1+1
Пользователь,,,,,,
Алиса,3.0,5.0,NaN,1,1.0,NaN
Боб,4.0,5.0,5.0,2,1.0,2.0
Чарли,3.0,5.0,4.0,2,NaN,3.0
Мэллори,5.0,5.0,4.0,2,NaN,3.0
Чак,1.0,1.0,NaN,5,5.0,NaN
Дейв,3.0,3.0,2.0,3,5.0,5.0
Ева,1.0,1.0,2.0,5,4.0,4.0
Пэгги,NaN,NaN,NaN,5,5.0,5.0


Пользователь,Алиса,Боб,Чарли,Мэллори,Чак,Дейв,Ева,Пэгги
Пользователь,,,,,,,,
Алиса,1.000000,0.769800,0.755929,0.787562,0.416025,0.592593,0.356966,0.192450
Боб,0.769800,1.000000,0.974707,0.974355,0.384308,0.744140,0.596462,0.333333
Чарли,0.755929,0.974707,1.000000,0.978059,0.314485,0.741930,0.603175,0.363696
Мэллори,0.787562,0.974355,0.978059,1.000000,0.312043,0.737558,0.566991,0.324785
Чак,0.416025,0.384308,0.314485,0.312043,1.000000,0.708784,0.821156,0.800641
Дейв,0.592593,0.744140,0.741930,0.737558,0.708784,1.000000,0.909914,0.833950
Ева,0.356966,0.596462,0.603175,0.566991,0.821156,0.909914,1.000000,0.945611
Пэгги,0.192450,0.333333,0.363696,0.324785,0.800641,0.833950,0.945611,1.000000


Пользователь
Алиса      1.000000
Мэллори    0.787562
Боб        0.769800
Чарли      0.755929
Дейв       0.592593
Чак        0.416025
Ева        0.356966
Пэгги      0.192450
Name: Алиса, dtype: float64

Рекомендации коллаборативной фильтрации для Алиса
	Дюна: прогнозируемая оценка = 3.65
	1+1: прогнозируемая оценка = 3.33


Рекомендации коллаборативной фильтрации для Чак
	1+1: прогнозируемая оценка = 4.03
	Дюна: прогнозируемая оценка = 2.95




In [57]:
display(user_similarity_df.loc['Алиса'].sort_values(ascending=False))

Пользователь
Алиса    1.000000
Боб      0.769800
Чарли    0.755929
Дейв     0.592593
Чак      0.416025
Ева      0.356966
Name: Алиса, dtype: float64

In [58]:
display(user_similarity_df.loc['Чак'].sort_values(ascending=False))

Пользователь
Чак      1.000000
Ева      0.821156
Дейв     0.708784
Алиса    0.416025
Боб      0.384308
Чарли    0.314485
Name: Чак, dtype: float64

In [9]:
df

,Интерстеллар,Звездный войны,Дюна,Побег из Шоушенка,Зеленая миля,1+1
Пользователь,,,,,,
Алиса,3,5,NaN,1,1.0,NaN
Боб,4,5,5.0,2,1.0,2.0
Чарли,3,5,4.0,2,NaN,3.0
Чак,1,1,NaN,5,5.0,NaN
Дейв,3,3,2.0,3,5.0,5.0
Ева,1,1,2.0,5,4.0,4.0


In [134]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

def get_als_recommendations(user, df, topk=2):
    users = df.index.tolist()
    movies = df.columns
    
    user_id = users.index(user) # id текущего пользователя

    user_item_matrix = csr_matrix(df.fillna(0).values.astype(np.float32))
    
    model = AlternatingLeastSquares(
        factors=2,
        iterations=1, # на нашем небольшом примере достаточно одной итерации
        random_state=42,
    )
    model.fit(user_item_matrix)

    # Получаем рекомендации (массив_id_объектов, массив_скоров)
    ids, scores = model.recommend(
        user_id,
        user_item_matrix[user_id],
        N=topk,
        filter_already_liked_items=True,
    )

    print(f"Рекомендации ALS для {user}")
    for movie_id, score in zip(ids, scores):
        print(f"\t {movies[movie_id]} (Уверенность модели: {score:.4f})")
    print('\n\n')
    return model

model = get_als_recommendations("Алиса", df)
model = get_als_recommendations("Чак", df)

100%|██████████| 1/1 [00:00<00:00, 3816.47it/s]


Рекомендации ALS для Алиса
	 Дюна (Уверенность модели: 0.6761)
	 1+1 (Уверенность модели: 0.4983)





100%|██████████| 1/1 [00:00<00:00, 3998.38it/s]

Рекомендации ALS для Чак
	 1+1 (Уверенность модели: 0.7193)
	 Дюна (Уверенность модели: 0.5228)





In [133]:
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans

def plot_emb(movie_vectors, df): 
    movie_names = df.columns.tolist()  # Наш список фильмов
    
    # 1. Автоматически делим фильмы на 2 кластера с помощью K-Means
    kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
    cluster_labels = kmeans.fit_predict(movie_vectors)
    
    df_plot = pd.DataFrame(
        {
            "X": movie_vectors[:, 0],
            "Y": movie_vectors[:, 1],
            "Название": movie_names,
            "Кластер": [
                f"Группа {c}" for c in cluster_labels
            ],  # преобразуем в текст для дискретной легенды
        }
    )
    
    fig = px.scatter(
        df_plot,
        x="X",
        y="Y",
        text="Название",
        color="Кластер",
        hover_name="Название",
        title="Интерактивная карта эмбеддингов",
        labels={"X": "Скрытый фактор 1", "Y": "Скрытый фактор 2"},
    )
    
    fig.update_traces(
        textposition="top center",
        marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")),  # Размер точек
    )
    
    fig.update_layout(
        xaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor="LightGray"),
        yaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor="LightGray"),
        plot_bgcolor="white",  # Белый фон
        width=900,
        height=600,
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="WhiteSmoke")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="WhiteSmoke")
    return fig

plot_emb(model.item_factors, df)

In [202]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

def convert_df(df_matrix):
    users = df_matrix.index.tolist()
    movies = df_matrix.columns.tolist()
    # Преобразуем в длинный формат (взаимодействия с оценкой > 0)
    interactions = []
    for u_idx, user in enumerate(users):
        for m_idx, movie in enumerate(movies):
            rating = df_matrix.loc[user, movie]
            if rating > 0:
                interactions.append((u_idx, m_idx, 1.0)) # 1.0 — факт позитивного взаимодействия
    
    df_train = pd.DataFrame(interactions, columns=["user_id", "item_id", "label"])
    return users, movies, df_train

# 2. Обертка данных в Dataset для PyTorch
class RecommendationDataset(Dataset):
    def __init__(self, dataframe):
        self.users = torch.tensor(dataframe["user_id"].values, dtype=torch.long)
        self.items = torch.tensor(dataframe["item_id"].values, dtype=torch.long)
        self.labels = torch.tensor(dataframe["label"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

# 3. Архитектура двухбашенной модели DSSM
class DSSMModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=8):
        super(DSSMModel, self).__init__()
        # Базовая башня Пользователя
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_nn = nn.Sequential(
            nn.Linear(embedding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8) # Финальный размер эмбеддинга (N-мерное пространство)
        )
        
        # Базовая башня Item
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.item_nn = nn.Sequential(
            nn.Linear(embedding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

    def forward(self, user_ids, item_ids):
        # Пропускаем пользователя через его башню
        u_emb = self.user_embedding(user_ids)
        u_vector = self.user_nn(u_emb)
        
        # Пропускаем товар через его башню
        i_emb = self.item_embedding(item_ids)
        i_vector = self.item_nn(i_emb)
        
        # Считаем косинусное сходство между векторами башен
        u_vector_norm = nn.functional.normalize(u_vector, p=2, dim=1)
        i_vector_norm = nn.functional.normalize(i_vector, p=2, dim=1)
        
        # Скалярное произведение строк (dot product)
        similarity = torch.sum(u_vector_norm * i_vector_norm, dim=1)
        return similarity

def fit_model(users, movies, dataloader, epochs, embedding_dim):
    # Инициализация
    model = DSSMModel(num_users=len(users), num_items=len(movies), embedding_dim=embedding_dim)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters())

    # Обучение модели
    model.train()
    for epoch in range(epochs):
        for batch_users, batch_items, batch_labels in dataloader:
            optimizer.zero_grad()
            predictions = model(batch_users, batch_items)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
    return model


def get_dssm_model_emb(df, epochs=10, embedding_dim=8):
    users, movies, df_train = convert_df(df)
    dataset = RecommendationDataset(df_train)
    dataloader = DataLoader(dataset, batch_size=1)
    model = fit_model(users, movies, dataloader, epochs, embedding_dim)
    
    model.eval()
    with torch.no_grad():
        all_item_ids = torch.tensor(list(range(len(movies))), dtype=torch.long)
        item_embs = model.item_embedding(all_item_ids)
        movie_embeddings = model.item_nn(item_embs).numpy()
    
    print(f"Форма матрицы эмбеддингов: {movie_embeddings.shape} (4 фильма, у каждого вектор из 8 признаков)")
    return movie_embeddings

plot_emb(get_dssm_model_emb(df, embedding_dim=8, epochs=10), df)

Форма матрицы эмбеддингов: (6, 8) (4 фильма, у каждого вектор из 8 признаков)


In [189]:
data = pd.read_csv('/Users/akuteuov/.cache/kagglehub/datasets/omarhanyy/imdb-top-1000/versions/1/IMDB top 1000.csv')
data

,Unnamed: 0,Title,Certificate,Duration,Genre,Rate,Metascore,Description,Cast,Info
0,0,1. The Shawshank Redemption (1994),R,142 min,Drama,9.3,80.0,"Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.","Director: Frank Darabont | Stars: Tim Robbins, Morgan Freeman, Bob Gunton, William Sadler","Votes: 2,295,987 | Gross: $28.34M"
1,1,2. The Godfather (1972),R,175 min,"Crime, Drama",9.2,100.0,The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son.,"Director: Francis Ford Coppola | Stars: Marlon Brando, Al Pacino, James Caan, Diane Keaton","Votes: 1,584,782 | Gross: $134.97M"
2,2,3. The Dark Knight (2008),PG-13,152 min,"Action, Crime, Drama",9.0,84.0,"When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.","Director: Christopher Nolan | Stars: Christian Bale, Heath Ledger, Aaron Eckhart, Michael Caine","Votes: 2,260,649 | Gross: $534.86M"
3,3,4. The Godfather: Part II (1974),R,202 min,"Crime, Drama",9.0,90.0,"The early life and career of Vito Corleone in 1920s New York City is portrayed, while his son, Michael, expands and tightens his grip on the family crime syndicate.","Director: Francis Ford Coppola | Stars: Al Pacino, Robert De Niro, Robert Duvall, Diane Keaton","Votes: 1,107,253 | Gross: $57.30M"
4,4,5. The Lord of the Rings: The Return of the King (2003),PG-13,201 min,"Action, Adventure, Drama",8.9,94.0,Gandalf and Aragorn lead the World of Men against Sauron's army to draw his gaze from Frodo and Sam as they approach Mount Doom with the One Ring.,"Director: Peter Jackson | Stars: Elijah Wood, Viggo Mortensen, Ian McKellen, Orlando Bloom","Votes: 1,614,369 | Gross: $377.85M"
...,...,...,...,...,...,...,...,...,...,...
995,995,398. Scent of a Woman (1992),R,156 min,Drama,8.0,NaN,"A prep school student needing money agrees to ""babysit"" a blind man, but the job is not at all what he anticipated.","Director: Martin Brest | Stars: Al Pacino, Chris O'Donnell, James Rebhorn, Gabrielle Anwar","Votes: 256,515 | Gross: $63.90M"
996,996,399. Aladdin (1992),G,90 min,"Animation, Adventure, Comedy",8.0,86.0,A kindhearted street urchin and a power-hungry Grand Vizier vie for a magic lamp that has the power to make their deepest wishes come true.,"Directors: Ron Clements, John Musker | Stars: Scott Weinger, Robin Williams, Linda Larkin, Jonathan Freeman","Votes: 367,489 | Gross: $217.35M"
997,997,400. JFK (1991),R,189 min,"Drama, History, Thriller",8.0,72.0,New Orleans District Attorney Jim Garrison discovers there's more to the Kennedy assassination than the official story.,"Director: Oliver Stone | Stars: Kevin Costner, Gary Oldman, Jack Lemmon, Walter Matthau","Votes: 139,634 | Gross: $70.41M"
998,998,301. Nights of Cabiria (1957),Not Rated,110 min,Drama,8.1,NaN,A waifish prostitute wanders the streets of Rome looking for true love but finds only heartbreak.,"Director: Federico Fellini | Stars: Giulietta Masina, François Périer, Franca Marzi, Dorian Gray","Votes: 42,160 | Gross: $0.75M"


In [191]:
data['Title'].shape

(1000,)

In [192]:
data['Title'].nunique()

398